In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from typing import Tuple
import os
import sys
%load_ext autoreload
%autoreload 2

load_dotenv()
pyAPES_main_folder = os.getenv('PYAPES_PATH')
sys.path.append(pyAPES_main_folder)

%matplotlib widget

In [ ]:
from pyAPES.utils.utilities import central_diff, tridiag, smooth

# Diagnosing closure_1_model_U

Why the wind speed calculation fails near ground



# Reynolds averaged Navier-Stokes 

Velocity has three components: $u$ (along mean wind, $x$), $v$ (crosswind, $y$), $w$ (vertical, $z$). We only need the budget of $u$, because that is the component the wind profile describes. Written out in full:

$$\underbrace{\frac{\partial u}{\partial t}}_{\text{storage}} + \underbrace{u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} + w\frac{\partial u}{\partial z}}_{\text{advection: wind carries its own momentum}} = \underbrace{-\frac{1}{\rho}\frac{\partial p}{\partial x}}_{\text{pressure force}} + \underbrace{\nu\left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} + \frac{\partial^2 u}{\partial z^2}\right)}_{\text{molecular friction}} + \underbrace{F_\mathrm{ext}}_\text{Drag force, see below}$$

Then let's assume mass conservation for incrompessible flow (good enough when $u < 100 \frac{m}{s}$ and we are near ground )

$$\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} + \frac{\partial w}{\partial z} = 0$$

**One algebraic step before averaging.** Rewrite the advection in *flux form* using the product rule backwards:

$$\frac{\partial (uu)}{\partial x} + \frac{\partial (uv)}{\partial y} + \frac{\partial (uw)}{\partial z} = u\frac{\partial u}{\partial x} + v\frac{\partial u}{\partial y} + w\frac{\partial u}{\partial z} + u\underbrace{\left(\frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} + \frac{\partial w}{\partial z}\right)}_{=\,0 \text{ by mass conservation}}$$

so we get
$$\underbrace{\frac{\partial u}{\partial t}}_{\text{storage}} + \underbrace{\frac{\partial (uu)}{\partial x} + \frac{\partial (uv)}{\partial y} + \frac{\partial (uw)}{\partial z}}_{\text{NB! this changed. advection=divergence}} = \underbrace{-\frac{1}{\rho}\frac{\partial p}{\partial x}}_\text{pressure force} +  \underbrace{\nu\left(\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} + \frac{\partial^2 u}{\partial z^2}\right)}_{\text{molecular friction}} + \underbrace{F_\mathrm{ext}}_\text{drag force}$$


So advection = divergence of the momentum fluxes $uu$, $uv$, $uw$. 

## Apply Reynolds averaging to NS 

**Assumptions**

- **(H1)** Stationary: time-averaged quantities do not change, $\partial(\;\overline{\cdot}\;)/\partial t = 0$
- **(H2)** Horizontally homogeneous: averaged quantities depend only on $z$, so $\partial(\;\overline{\cdot}\;)/\partial x = \partial(\;\overline{\cdot}\;)/\partial y = 0$
- **(H3, H4)** Mean vertical and crosswind velocities vanish: $\overline{W}=0$ (from mass conservation + H2 + solid ground), $\overline{V}=0$ ($x$-axis chosen along the mean wind)
- **(H5)** High Reynolds number: molecular friction negligible compared to turbulent transport (its effect at the surface is absorbed into $z_0$ later)

**Reynolds decomposition.** Each instantaneous field = mean + fluctuation:

$$u = \overline{U}(z) + u', \qquad v = 0 + v', \qquad w = 0 + w'$$

with the averaging rules: $\overline{u'} = \overline{v'} = \overline{w'} = 0$, and the average of (mean × fluctuation) is zero.

**Average the three flux terms.** 

$$\overline{uu} = \overline{(\overline{U}+u')(\overline{U}+u')} = \overline{U}^2 + 2\,\overline{U}\,\underbrace{\overline{u'}}_{=0} + \overline{u'u'} = \overline{U}^2 + \overline{u'u'}$$

$$\overline{uv} = \overline{(\overline{U}+u')(0+v')} = \overline{U}\,\underbrace{\overline{v'}}_{=0} + \overline{u'v'} = \overline{u'v'}$$

$$\overline{uw} = \overline{(\overline{U}+u')(0+w')} = \overline{U}\,\underbrace{\overline{w'}}_{=0} + \overline{u'w'} = \overline{u'w'}$$

**Substitute Reynolds decomposition into Navier stokes** 

$$\frac{\partial \overline{U}}{\partial t} + \frac{\partial}{\partial x}\big(\overline{U}^2 + \overline{u'u'}\big) + \frac{\partial}{\partial y}\big(\overline{u'v'}\big) + \frac{\partial}{\partial z}\big(\overline{u'w'}\big) = -\frac{1}{\rho}\frac{\partial \overline{P}}{\partial x} + \text{(viscous)} + F_\mathrm{ext}$$

**Now apply H1–H2,H5** 
 - Storage term $\frac{\partial U}{\partial t}$ vanishes(H1). 
 - $x$- and $y$-derivatives of the averaged fluxes die (H2): 
    - $\partial(\overline{U}^2 + \overline{u'u'})/\partial x = 0$
    - $\partial\,\overline{u'v'}/\partial y = 0$. 
    - Molecular friction is neglected (H5). 
    
We are left with

$$\frac{d\,\overline{u'w'}}{dz} = -\frac{1}{\rho}\frac{d\overline{P}}{dx} + F_\mathrm{ext}$$

$\overline{u'w'}$ is the correlation between vertical motion and horizontal-momentum anomaly: when updrafts ($w'>0$) systematically carry slow air ($u'<0$), momentum flows downward. if we define

$$\tau(z) \equiv -\rho\,\overline{u'w'}$$

we get the 1D momentum budget:

$$\frac{d\tau}{dz} = \frac{d\overline{P}}{dx} + F_\mathrm{ext}$$

With no pressure gradient: $d\tau/dz = 0$ — the **constant-flux layer**. Inside a canopy, plant elements exert form drag on the air. Intuitively one can think that the some of the momentum is transferred into motion of trees, branches and leaves. This drag is proportional to the amount of leaves, density of air, and wind speed

$$\frac{d\tau}{dz} =  \underbrace{C_d\, \mathrm{LAD}(z)\, \overline{U}^2}_\text{$F_\mathrm{ext}$ in this case} + \frac{d\overline{P}}{dx}$$

which is exactly the equation `closure_1_model_u` solves.

## Logarithmic wind profile

Assumption: near ground momentum flux is nearly constant 
$$
\begin{align*}
\frac{d\tau}{dz} &= 0 \\
\longrightarrow \tau(z) &= \mathrm{constant} \\
\tau(z) &:= \tau_\mathrm{surface} 
\end{align*}
$$

if we define $\tau_\mathrm{surface} = \rho u^{*2}$ (this is the definition of friction velocity $u^*$)

since profile is constant also everywhere else  

$$\tau(z) = \rho u^{*2}$$

**Closure model and mixing length hypothesis**

If we further assume first order closure scheme that relates $\overline{u'w'}$ to mean horizontal windspeed $\overline{U}$ (later referred to just U) we get

$$ \tau = \rho K_m \frac{dU}{dz} $$

And with the Prandtl mixing length theory where we treat eddy viscosity $K_m$ analogously to mean free path in thermodynamics (we assume a fluid parcel conserves it properties for some characteristic length $l$ before mixing with the surrounding fluid) we can parametrize 

$$K_m = l^2\left| \frac{dU}{dz} \right|$$

We finally get

$$ 
\begin{align*}
\tau &= \rho l^2 \left( \frac{dU}{dz} \right)^2 \\
\rho u^{*2} &= \rho l^2 \left( \frac{dU}{dz} \right)^2 \\
\frac{dU}{dz} &= \frac{u^*}{l}
\end{align*}
$$

if we set $l = \kappa z$ and use separation of variables to solve the differential equation we get the logarithmic wind profile

$$ U(z) = \frac{u^*}{\kappa} \mathrm{ln}\left(\frac{z}{z_0}\right)$$

where $z_0$ is the roughness length (height where the U profile hits zero) and comes from substituting $U(z_0)=0$ and solving for the integration constant one gets with the separation of variables

# Current implementation of `closure_1_model_U`

In [ ]:
VON_KARMAN = 0.4
EPS = 1e-16
def mixing_length(z: np.ndarray, h: float, d: float, l_min: float=None) -> np.ndarray:
    """
    Computes turbulend mixing length. The l_mix is assumed linear above the canopy, constant within and
    decreases linearly close the ground (below z< alpha*h/VON_KARMAN)
    
    References:
        Juang, J.-Y., Katul, G.G., Siqueira, M.B., Stoy, P.C., McCarthy, H.R., 2008.
        Investigating a hierarchy of Eulerian closure models for scalar transfer inside
        forested canopies. Boundary-Layer Meteorology 128, 1–32.    
    Args:
        z (array): [m], computation grid, constant increment
        h (float): [m], canopy height
        d (float): [m], displacement height
        l_min (float): [m], set to finite value at ground
    
    Returns:
        (np.ndarray):
            lmix (array): [m], turbulent mixing length

    """
    dz = z[1] - z[0]

    if not l_min:
        l_min = dz/2.0
    alpha = (h - d)*VON_KARMAN / (h + EPS)
    I_F = np.sign(z - h) + 1.0
    l_mix = alpha*h*(1 - I_F / 2) + (I_F / 2) * (VON_KARMAN*(z - d))

    sc = (alpha*h) / VON_KARMAN
    ix = np.where(z < sc)
    l_mix[ix] = VON_KARMAN*(z[ix] + dz / 2)
    l_mix = l_mix + l_min

    return l_mix

In [ ]:
def closure_1_model_U(z: np.ndarray, Cd: float, lad: np.ndarray, hc: float,
                      Utop: float, Ubot: float, dPdx: float = 0.0, lbc_flux: bool = None,
                      U_ini: np.array = None) -> Tuple:
    """
    Mean velocity profile, shear stress and eddy diffusivity within and above 
    horizontally homogenous plant canopies using 1st order closure. Accounts 
    for horizontal pressure gradient force dPdx, assumes neutral diabatic stability.
    Solves displacement height as centroid of drag force.

    Args:
       z - height [m]], constant increments
       Cd - drag coefficient (typical range 0.1 - 0.3) [-]
       lad - plant area density, 1-sided [m2 m-3]
       hc - canopy height [m]
       Utop - U /u* [-] upper boundary
       Ubot - U /u* [-] at ground (0.0 for no-slip)
       dPdx - u* -normalized horizontal pressure gradient
       lbc_flux - True sets lower BC to zero flux

    Returns:
        (tuple):
            tau (array): u* -normalized momentum flux
            U (array): u* normalized mean wind speed [-]]
            Km (array): eddy diffusivity for momentum [m2 s-1]
            l_mix (array): mixing length [m]
            d (float): zero-plane displacement height [m]
            zo (float): roughness lenght for momentum [m]

    """

    lad = 0.5*lad  # frontal plant-area density is half of one-sided
    # dz = z[1] - z[2]
    dz = z[2] - z[1]
    N = len(z)
    if U_ini is None:
        U = np.linspace(Ubot, Utop, N)
    else:
        U = U_ini.copy()

    nn1 = max(2, np.floor(N/20))  # window for moving average smoothing

    # --- Start iterative solution
    err = 999.9
    iter_max = 200
    eps1 = 0.5
    dPdx_m = 0.0

    iter_no = 0.0

    while err > 0.001 and iter_no < iter_max:
        iter_no += 1
        Fd = Cd*lad*U**2  # drag force
        d = sum(z*Fd) / (sum(Fd) + EPS)  # displacement height
        l_mix = mixing_length(z, hc, d, l_min=0.01)  # m
        # --- dU/dz [m-1]
        y = central_diff(U, dz)

        # --- eddy diffusivity & shear stress
        Km = l_mix**2*abs(y)
        tau = -Km * y

        # ------ Set the elements of the Tri-diagonal Matrix
        a1 = -Km
        a2 = central_diff(-Km, dz)
        a3 = Cd*lad*U

        upd = (a1 / (dz*dz) + a2 / (2*dz))  # upper diagonal
        dia = (-a1*2 / (dz*dz) + a3)  # diagonal
        lod = (a1 / (dz*dz) - a2 / (2*dz))  # subdiagonal
        rhs = np.ones(N) * dPdx  # _m ???

        # upper BC
        upd[-1] = 0.
        dia[-1] = 1.
        lod[-1] = 0.
        rhs[-1] = Utop

        if not lbc_flux:  # --- lower BC, fixed Ubot
            upd[0] = 0.
            dia[0] = 1.
            lod[0] = 0.
            rhs[0] = Ubot
        else:  # --- lower BC, flux-based
            upd[0] = -1.
            dia[0] = 1.
            lod[0] = 0.
            rhs[0] = 0.  # zero-flux bc
            # rhs[0] = lbc_flux

        # --- call tridiagonal solver
        Un = tridiag(lod, dia, upd, rhs)

        err = max(abs(Un - U))

        # --- Use successive relaxations in iterations
        U = eps1*Un + (1.0 - eps1)*U
        dPdx_m = eps1*dPdx + (1.0 - eps1)*dPdx_m  # ???

    y_orig = y
    # ---- return values
    tau_orig = tau
    tau = tau / tau[-1]  # normalized shear stress
    zo = (z[-1] - d)*np.exp(-0.4*U[-1])  # roughness length

    y = central_diff(U, dz)
    Kmr = l_mix**2 * abs(y)  # eddy diffusivity
    Km = smooth(Kmr, nn1)

    # --- for testing ----
#    plt.figure(101)
#    plt.subplot(221); plt.plot(Un, z, 'r-'); plt.title('U')
#    plt.subplot(222); plt.plot(y, z, 'b-'); plt.title('dUdz')
#    plt.subplot(223); plt.plot(l_mix, z, 'r-'); plt.title('l mix')
#    plt.subplot(224); plt.plot(Km, z, 'r-', Kmr, z, 'b-'); plt.title('Km')

    return tau, U, Km, l_mix, d, zo, tau_orig, y_orig

## Set properties for open snow field

In [ ]:
lad = 0
z_orig = np.linspace(0,25,101)
dz = z_orig[1] - z_orig[0]
z_orig_midpoint = z_orig[:-1] + dz/2


hc = 1e-2
Cd = 0.2
dPdx = 0.0
z0 = 0.001 # Roughness length for snow surface
U_top = 5
U_bot = 0.01

# Calculate u* from the logarithmic wind profile
u_star = VON_KARMAN*U_top/np.log(z_orig_midpoint[-1]/z0+EPS)

# Scale U_top and U_bot for closure_1_model_U
U_top_norm = U_top / u_star
U_bot_norm = U_bot / u_star


## Solve U with `closure_1_model_U`

In [ ]:
tau, U , Km, l_mix, d, z0_c, tau_orig, y_orig = closure_1_model_U(z_orig_midpoint, Cd, lad, hc, U_top_norm, U_bot_norm, dPdx)

In [ ]:
z0_c

## Solve U from logarithmic wind profile

In [ ]:
U_log = u_star / VON_KARMAN * np.log(z_orig_midpoint / z0)

In [ ]:
fig, axs = plt.subplots(figsize=(16, 6), ncols=5, nrows=1)
fig.subplots_adjust(wspace=0.4, hspace=0.3)
axs = axs.flatten()
axs[0].plot(U*u_star, z_orig_midpoint, 'k-o', label='closure_1_model_U')
axs[0].plot(U_log, z_orig_midpoint, 'r--o', label='logarithmic profile (goal)')
axs[0].legend(bbox_to_anchor=(0.0, 1.15), loc='upper left')

#logarithmic y-axis
axs[1].semilogy(U*u_star, z_orig_midpoint, 'k-o')
axs[1].semilogy(U_log, z_orig_midpoint, 'r--o')

axs[2].plot(tau, z_orig_midpoint, 'k-o')
axs[3].plot(Km, z_orig_midpoint, 'k-o')

axs[4].plot(l_mix, z_orig_midpoint, 'k-o')

xlabels = ['U [m/s]','U [m/s]',r'$\tau$ [Pa]', 'K$_m$ [m$^2$/s]', '$l_{mix}$ [m]']
for i,ax in enumerate(axs):
    ax.set_xlabel(xlabels[i])
    ax.set_ylabel('z [m]')


# The problem

- **P1** Our $z$ grid starts at $0 \mathrm{m}$ even though theory does not say anything about $U(z=0\ldots z_0)$ <br />
- **P2** Assuming $U_\mathrm{bot}$ is small (in the code we use $U_\mathrm{bot} = 0.01$) leads to lower $U$ at the bottom <br />
- **P3** Most of the change in U profile happens where there are only couple of grid points 
- **P4** Weird formula for $l$? Should it be $\max\{l, l_\mathrm{min}\}$, not $l+l_\mathrm{min}$ which always increases $l$ at the bottom.

$\longrightarrow$ Snow model gets wrong conductance

# Possible fixes

- **F1** Use mid points of the element such that 0 is not in $z$ grid
- **F2** decrease $\Delta z$ (might be possible since `closure_1_model_U` converges fast)
- **F3** Calculate $U_\mathrm{bot}$ from logarithmic profile on open surfaces
    - Maybe additional control flag to micrometeorology parameters?
- **F4** Fix `mixing_length` function
    - Maximum instead of additive $l_\mathrm{min}$
    - if $z$ still has zero in it set $l = \kappa (z+z_0)$ for the lowest elements
        - $z_0$ is already a parameter of the micrometeorology module


In [ ]:
def mixing_length_new(z: np.ndarray, h: float, d: float, z0: float, l_min: float=None) -> np.ndarray:
    """
    Computes turbulend mixing length. The l_mix is assumed linear above the canopy, constant within and
    decreases linearly close the ground (below z< alpha*h/VON_KARMAN)
    
    References:
        Juang, J.-Y., Katul, G.G., Siqueira, M.B., Stoy, P.C., McCarthy, H.R., 2008.
        Investigating a hierarchy of Eulerian closure models for scalar transfer inside
        forested canopies. Boundary-Layer Meteorology 128, 1–32.    
    Args:
        z (array): [m], computation grid, constant increment
        h (float): [m], canopy height
        d (float): [m], displacement height
        z0 (float): [m], roughness length NEW PARAMETER
        l_min (float): [m], set to finite value at ground
    
    Returns:
        (np.ndarray):
            lmix (array): [m], turbulent mixing length

    """
    dz = z[1] - z[0]

    if l_min is None:
        l_min = VON_KARMAN * z0

    if h < 3*dz:  # THIS CHANGED Open ground assume l = kappa * (z+z0) -> canopy nodes not resolved
        return np.maximum(VON_KARMAN*(z+z0), l_min)
    alpha = (h - d)*VON_KARMAN / (h + EPS)
    I_F = np.sign(z - h) + 1.0
    l_mix = alpha*h*(1 - I_F / 2) + (I_F / 2) * (VON_KARMAN*(z - d))

    sc = (alpha*h) / VON_KARMAN
    ix = np.where(z < sc)
    l_mix[ix] = VON_KARMAN*(z[ix] + z0) # this changed
    l_mix = np.maximum(l_mix,l_min) # this changed

    return l_mix

In [ ]:
def closure_1_model_U_new(z: np.ndarray, z0: float, Cd: float, lad: np.ndarray, hc: float,
                      Utop: float, Ubot: float, dPdx: float = 0.0, lbc_flux: bool = None,
                      U_ini: np.array = None) -> Tuple:
    """
    Mean velocity profile, shear stress and eddy diffusivity within and above 
    horizontally homogenous plant canopies using 1st order closure. Accounts 
    for horizontal pressure gradient force dPdx, assumes neutral diabatic stability.
    Solves displacement height as centroid of drag force.

    Args:
       z - height [m]], constant increments
       z0 - roughness length [m] NEW PARAMETER
       Cd - drag coefficient (typical range 0.1 - 0.3) [-]
       lad - plant area density, 1-sided [m2 m-3]
       hc - canopy height [m]
       Utop - U /u* [-] upper boundary
       Ubot - U /u* [-] at ground (0.0 for no-slip)
       dPdx - u* -normalized horizontal pressure gradient
       lbc_flux - True sets lower BC to zero flux

    Returns:
        (tuple):
            tau (array): u* -normalized momentum flux
            U (array): u* normalized mean wind speed [-]]
            Km (array): u* normalized eddy diffusivity for momentum [m] #IS THIS CORRECT?
            l_mix (array): mixing length [m]
            d (float): zero-plane displacement height [m]
            zo (float): roughness lenght for momentum [m]

    """

    lad = 0.5*lad  # frontal plant-area density is half of one-sided
    # dz = z[1] - z[2]
    #dz = z[2] - z[1]
    dz = np.zeros_like(z)
    dz[1:-1] = (z[2:] - z[:-2]) / 2.0  
    dz[0] = z[1] - z[0]
    dz[-1] = z[-1] - z[-2]
    N = len(z)
    if U_ini is None:
        U = np.linspace(Ubot, Utop, N)
    else:
        U = U_ini.copy()

    nn1 = max(2, np.floor(N/20))  # window for moving average smoothing

    # --- Start iterative solution
    err = 999.9
    iter_max = 200
    eps1 = 0.5
    dPdx_m = 0.0

    iter_no = 0.0

    while err > 0.001 and iter_no < iter_max:
        iter_no += 1
        Fd = Cd*lad*U**2  # drag force
        d = sum(z*Fd) / (sum(Fd) + EPS)  # displacement height
        l_mix = mixing_length_new(z, hc, d, z0, l_min=VON_KARMAN * z0)  # m THIS CHANGED

        # --- dU/dz [m-1]
        y = central_diff(U, dz)

        # --- eddy diffusivity & shear stress
        Km = l_mix**2*abs(y)
        tau = -Km * y

        # ------ Set the elements of the Tri-diagonal Matrix
        a1 = -Km
        a2 = central_diff(-Km, dz)
        a3 = Cd*lad*U

        upd = (a1 / (dz*dz) + a2 / (2*dz))  # upper diagonal
        dia = (-a1*2 / (dz*dz) + a3)  # diagonal
        lod = (a1 / (dz*dz) - a2 / (2*dz))  # subdiagonal
        rhs = np.ones(N) * dPdx  # _m ???

        # upper BC
        upd[-1] = 0.
        dia[-1] = 1.
        lod[-1] = 0.
        rhs[-1] = Utop

        if not lbc_flux:  # --- lower BC, fixed Ubot
            upd[0] = 0.
            dia[0] = 1.
            lod[0] = 0.
            rhs[0] = Ubot
        else:  # --- lower BC, flux-based
            upd[0] = -1.
            dia[0] = 1.
            lod[0] = 0.
            rhs[0] = 0.  # zero-flux bc
            # rhs[0] = lbc_flux

        # --- call tridiagonal solver
        Un = tridiag(lod, dia, upd, rhs)

        err = max(abs(Un - U))

        # --- Use successive relaxations in iterations
        U = eps1*Un + (1.0 - eps1)*U
        dPdx_m = eps1*dPdx + (1.0 - eps1)*dPdx_m  # ???

    y_orig = y
    # ---- return values
    tau_orig = tau
    tau = tau / tau[-1]  # normalized shear stress
    #zo = (z[-1] - d)*np.exp(-0.4*U[-1])  # roughness length

    y = central_diff(U, dz)
    Kmr = l_mix**2 * abs(y)  # eddy diffusivity
    Km = smooth(Kmr, nn1)

    # --- for testing ----
#    plt.figure(101)
#    plt.subplot(221); plt.plot(Un, z, 'r-'); plt.title('U')
#    plt.subplot(222); plt.plot(y, z, 'b-'); plt.title('dUdz')
#    plt.subplot(223); plt.plot(l_mix, z, 'r-'); plt.title('l mix')
#    plt.subplot(224); plt.plot(Km, z, 'r-', Kmr, z, 'b-'); plt.title('Km')

    return tau, U, Km, l_mix, d, z0, tau_orig, y_orig

# First look at the changes to $l_{mix}$ and $U$ with old grid

In [ ]:
z_new = z_orig_midpoint

hc = 0
Cd = 0.2
dPdx = 0.0
z0 = 0.001 # Roughness length for snow surface
U_top = 5

# Calculate u* from the logarithmic wind profile
u_star = VON_KARMAN*U_top/np.log(z_new[-1]/z0)

# NEW U_BOT
U_bot_new = u_star / VON_KARMAN * np.log(z_new[0]/z0)  # Assuming no-slip at the surface

# Scale U_top and U_bot for closure_1_model_U
U_top_norm = U_top / u_star
U_bot_norm_new = U_bot_new / u_star

In [ ]:
U_bot_new

In [ ]:
tau_new, U_new , Km_new, l_mix_new, d, z0, tau_orig, y_orig = closure_1_model_U_new(z_new, z0, Cd, lad, hc, U_top_norm, U_bot_norm_new, dPdx)

In [ ]:
fig, axs = plt.subplots(figsize=(16, 6), ncols=5, nrows=1)
fig.subplots_adjust(wspace=0.4, hspace=0.3)
axs = axs.flatten()
axs[0].plot(U*u_star, z_orig_midpoint, 'k-o', label='closure_1_model_U')
axs[0].plot(U_new*u_star, z_new, 'b-o', label='closure_1_model_U (new)')
axs[0].plot(U_log, z_orig_midpoint, 'r--o', label='logarithmic profile (goal)', markersize=4)
axs[0].legend(bbox_to_anchor=(-0.1, 1.15), loc='upper left')

#logarithmic y-axis
axs[1].semilogy(U*u_star, z_orig_midpoint, 'k-o')
axs[1].semilogy(U_new*u_star, z_new, 'b-o')
axs[1].semilogy(U_log, z_orig_midpoint, 'r--o', markersize=4)

axs[2].plot(tau_new, z_new, 'b-o', label='closure_1_model_U (new)')
axs[2].plot(tau, z_orig_midpoint, 'k-o', label='closure_1_model_U')
axs[3].plot(Km_new*u_star, z_new, 'b-o', label='closure_1_model_U (new)')
axs[3].plot(Km*u_star, z_orig_midpoint, 'k-o', label='closure_1_model_U')

axs[4].semilogy(l_mix_new, z_new, 'b-o', label='closure_1_model_U (new)')
axs[4].semilogy(l_mix, z_orig_midpoint, 'k-o', label='closure_1_model_U', markersize=4)

xlabels = ['U [m/s]','U [m/s]',r'$\tau$ [Pa]', 'K$_m$ [m$^2$/s]', '$l_{mix}$ [m]']
for i,ax in enumerate(axs):
    ax.set_xlabel(xlabels[i])
    ax.set_ylabel('z [m]')


# Second attempt: decrease $\Delta z$, keep old $U_\mathrm{bot}$

In [ ]:
z_new = np.linspace(0,25,12501*2)
dz = z_new[1] - z_new[0]
hc = 0
Cd = 0.2
dPdx = 0.0
z0 = 0.001 # Roughness length for snow surface
U_top = 5

# Calculate u* from the logarithmic wind profile
u_star = VON_KARMAN*U_top/np.log(z_new[-1]/z0)

# NEW U_BOT
U_bot_new = 0.00

# Scale U_top and U_bot for closure_1_model_U
U_top_norm = U_top / u_star
U_bot_norm_new = U_bot_new / u_star

In [ ]:
tau_new, U_new , Km_new, l_mix_new, d, z0, tau_orig, y_orig = closure_1_model_U_new(z_new, z0, Cd, lad, hc, U_top_norm, U_bot_norm_new, dPdx)

In [ ]:
fig, axs = plt.subplots(figsize=(16, 6), ncols=5, nrows=1)
fig.subplots_adjust(wspace=0.4, hspace=0.3)
axs = axs.flatten()
axs[0].plot(U*u_star, z_orig_midpoint, 'k-o', label='closure_1_model_U')
axs[0].plot(U_new*u_star, z_new, 'b-o', label='closure_1_model_U (new)')
axs[0].plot(U_log, z_orig_midpoint, 'r--o', label='logarithmic profile (goal)')
axs[0].legend(bbox_to_anchor=(-0.1, 1.15), loc='upper left')

#logarithmic y-axis
axs[1].semilogy(U*u_star, z_orig_midpoint, 'k-o')
axs[1].semilogy(U_new*u_star, z_new, 'b-o')
axs[1].semilogy(U_log, z_orig_midpoint, 'r--o')

axs[2].plot(tau_new, z_new, 'b-o', label='closure_1_model_U (new)')
axs[2].plot(tau, z_orig_midpoint, 'k-o', label='closure_1_model_U')
axs[3].plot(Km_new*u_star, z_new, 'b-o', label='closure_1_model_U (new)')
axs[3].plot(Km*u_star, z_orig_midpoint, 'k-o', label='closure_1_model_U')

axs[4].plot(l_mix_new, z_new, 'b-o', label='closure_1_model_U (new)')
axs[4].plot(l_mix, z_orig_midpoint, 'k-o', label='closure_1_model_U', markersize=4)

xlabels = ['U [m/s]','U [m/s]',r'$\tau$ [Pa]', 'K$_m$ [m$^2$/s]', '$l_{mix}$ [m]']
for i,ax in enumerate(axs):
    ax.set_xlabel(xlabels[i])
    ax.set_ylabel('z [m]')
